# `parse_question` — Brique 2 : parser une question utilisateur

Référence : [`docs/06_question_parsing.md`](../docs/06_question_parsing.md).

Module testé : [`src/docpipeline/question_parsing/question_parsing.py`](../src/docpipeline/question_parsing/question_parsing.py).

## Ce que fait `parse_question(question, document_type)`

Transforme une question utilisateur en un JSON structuré (`list[dict]`, drop-in compat avec `src/question/understand_question`). Chaque entrée contient :

| Section | Champs |
|---|---|
| `retrieval`  | `main_query`, `page_hint`, `section_hint`, `layout_hint`, `document_hint`, `anchor_keywords` |
| `generation` | `original_question`, `expected_answer_shape`, `format_constraint`, `disambiguation`, `must_distinguish` |
| `_meta`      | `intent` (`qa` / `summarization` / `translation`), `document_type`, `bricks_active` |

**Champs alignés sur le chapitre 6** :
- `intent` : `qa` (défaut), `summarization`, `translation` — chaque intent route vers un pipeline différent.
- `expected_answer_shape` : `amount` / `date` / `list` / `table` / `text` (+ extensions `boolean` / `entity` / `numeric`) — détecté à parsing time depuis la formulation seule, pilote la mise en forme côté generation.
- Le JSON ne contient que des champs populés (pas de `null`).

Approche : regex / heuristique uniquement (pas de LLM, conforme à `CLAUDE.md`). La banque de cas ci-dessous valide les couvertures sur 12 questions représentatives.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
from docpipeline.question_parsing import parse_question

# Banque de 12 cas couvrant : 3 intents × 7 shapes × hints structurels variés.
# Pour chaque cas : (question, doc_type, intent_attendu, shape_attendue).
CASES = [
    ('What is the premium amount on this contract? Page 5.',                'pdf',   'qa',            'amount'),
    ('When does coverage start? Format YYYY-MM-DD.',                        'pdf',   'qa',            'date'),
    ('List all the obligations of the seller.',                             'pdf',   'qa',            'list'),
    ('Compare the indemnification and liability caps in the latest version.','pdf',  'qa',            'table'),
    ('Explain the warranty section.',                                       'pdf',   'qa',            'text'),
    ('Does this contract include a non-compete clause?',                    'pdf',   'qa',            'boolean'),
    ('Who is the insured party?',                                           'pdf',   'qa',            'entity'),
    ('Summarize the exclusions section.',                                   'pdf',   'summarization', 'text'),
    ('Translate the warranty clause to French.',                            'pdf',   'translation',   'text'),
    ('Combien coute la prime annuelle ?',                                   'pdf',   'qa',            'amount'),
    ('Look in the exclusions section for the flooding clause.',             'pdf',   'qa',            'text'),
    ('Does article L131-1 apply, in the Limits section page 5?',            'pdf',   'qa',            'boolean'),
]

rows = []
for q, doc, exp_intent, exp_shape in CASES:
    plan = parse_question(q, document_type=doc)[0]
    intent = plan['_meta']['intent']
    shape  = plan['generation'].get('expected_answer_shape', '?')
    bricks = ','.join(plan['_meta']['bricks_active']) or '-'
    ok     = (intent == exp_intent and shape == exp_shape)
    rows.append({
        'OK':            'OK' if ok else 'FAIL',
        'intent':        intent,
        'shape':         shape,
        'exp_intent':    exp_intent,
        'exp_shape':     exp_shape,
        'bricks_active': bricks,
        'question':      q[:60],
    })

import pandas as pd
print(pd.DataFrame(rows).to_string(index=False))
print()
print('=== Exemple détaillé : la question la plus riche ===')
print(json.dumps(parse_question(
    "What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies."
)[0], indent=2, ensure_ascii=False))

OK        intent   shape    exp_intent exp_shape                                       bricks_active                                                     question
OK            qa  amount            qa    amount                              page_hint,answer_shape         What is the premium amount on this contract? Page 5.
OK            qa    date            qa      date                                 format,answer_shape                 When does coverage start? Format YYYY-MM-DD.
OK            qa    list            qa      list                                        answer_shape                      List all the obligations of the seller.
OK            qa   table            qa     table                          document_hint,answer_shape Compare the indemnification and liability caps in the latest
OK            qa    text            qa      text                           section_hint,answer_shape                                Explain the warranty section.
OK            qa boolean    